In [0]:
"""

===============================================================================
Procedure: Load Silver Layer (Bronze -> Silver)
===============================================================================
Script Purpose:
    This stored procedure performs the ETL (Extract, Transform, Load) process to 
    populate the 'silver' schema tables from the 'bronze' schema.
	Actions Performed:
		- Truncates Silver tables.
		- Inserts transformed and cleansed data from Bronze into Silver tables.
		

"""

In [0]:
#init 

catalog_name = "abhi_dwh_sql_based"
source_schema  = "bronze"
sink_schema = "silver"
table_name = "crm_cust_info"

In [0]:
#prepare the prerequists for the bronze layer,

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{sink_schema}")
print("Schema created --> ", catalog_name +"."+ sink_schema)

# Read the cust_info from the bronze layer 

In [0]:
df = spark.read.table(f"{catalog_name}.{source_schema}.{table_name}")



In [0]:
from pyspark.sql.functions import col,trim,upper, when ,row_number ,create_map,lit
from itertools import chain
gndr_map = {'F': 'Female','M': 'Male'}
expr = None
for k, v in gndr_map.items():
    expr = when(upper(col("cst_gndr")) == k, v) if expr is None else expr.when(upper(col("cst_gndr")) == k, v)
    

In [0]:
expr

## Transformation to clean the data

In [0]:
from pyspark.sql.functions import col,trim,upper, when ,row_number 
from pyspark.sql.window import Window

#select where customer id is not null 
df = df.filter(col('cst_id').isNotNull())

#take the latedt record for the customer since we are taking the customer info only 
window=Window.partitionBy('cst_id').orderBy(col('cst_create_date').desc())

df = df.withColumn('flag_last',row_number().over(window)).filter(col('flag_last')==1).drop('flag_last')

#trim the extra leading and trailing faces from the all columns
# df = df.select(*[trim(col(c)).alias(c) for c in df.columns])

#Above code will be find but in production we will use the below code becuase above code only works better with the string columns , otherwse spark might throw an type issues 
df = df.select(*[trim(col(c)).alias(c) if dict(df.dtypes)[c] == 'string' else col(c) for c in df.columns])


#convert the gender column to upper case and replace the values to 'M' and 'F' to 'Male' and 'Female'
#Basic
# df = df.withColumn('cst_gndr',
#     when(upper(col('cst_gndr'))=='F', 'Female')
#     .when(upper(col('cst_gndr'))=='M','Male')
#     .otherwise("n/a")
# )
#Adv
gndr_map = {'F': 'Female','M': 'Male'}
df = df.withColumn('cst_gndr',
                   when(upper(col('cst_gndr')).isin(list(gndr_map.keys())), upper(col('cst_gndr')).replace(gndr_map))
                   .otherwise("n/a")
                   )


#convert the marital status column to upper case and replace the values to 'S' and 'M' to 'Single' and 'Married'
#Basic
# df = df.withColumn('cst_marital_status',
#     when(upper(col('cst_marital_status'))=='S', 'Single')
#     .when(upper(col('cst_marital_status'))=='M','Married')
#     .otherwise("n/a")
# )
#Adv
marital_status_map = {'S': 'Single','M': 'Married'}
df = df.withColumn('cst_marital_status',
                   when(upper(col('cst_marital_status')).isin(list(marital_status_map.keys())), upper(col('cst_marital_status')).replace(marital_status_map))
                   .otherwise("n/a")
                   )

#


In [0]:
df.show()

# Write it to Silver Layer after cleansing 

In [0]:
df.write.mode("overwrite").saveAsTable(f"{catalog_name}.{sink_schema}.{table_name}")
